# 👥 Customer Retention Analysis — SwiftEats

Cohort retention matrix, RFM segmentation, churn risk, and CLV.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER','food_user')}:{os.getenv('DB_PASSWORD','')}"
    f"@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}"
    f"/{os.getenv('DB_NAME','food_delivery')}"
)
print("Connected ✓")

## 1. Cohort Retention Heatmap

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        WITH first_order AS (
            SELECT customer_id,
                   DATE_TRUNC('month', MIN(order_timestamp))::date AS cohort_month
            FROM orders WHERE order_status = 'Delivered' GROUP BY 1
        ),
        activity AS (
            SELECT fo.customer_id, fo.cohort_month,
                   (EXTRACT(YEAR FROM AGE(DATE_TRUNC('month',o.order_timestamp),
                       fo.cohort_month::TIMESTAMP))*12 +
                    EXTRACT(MONTH FROM AGE(DATE_TRUNC('month',o.order_timestamp),
                       fo.cohort_month::TIMESTAMP)))::INT AS mn
            FROM first_order fo JOIN orders o USING(customer_id)
            WHERE o.order_status = 'Delivered'
        ),
        sizes AS (SELECT cohort_month, COUNT(*) AS sz FROM first_order GROUP BY 1)
        SELECT TO_CHAR(a.cohort_month,'YYYY-MM') AS cohort, s.sz,
               a.mn AS month_num,
               ROUND(COUNT(DISTINCT a.customer_id)*100.0/s.sz,1) AS retention_pct
        FROM activity a JOIN sizes s USING(cohort_month)
        WHERE a.mn BETWEEN 0 AND 6
        GROUP BY a.cohort_month, s.sz, a.mn
        ORDER BY a.cohort_month, a.mn
    '''), conn)

pivot = df.pivot(index='cohort', columns='month_num', values='retention_pct')
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Retention %'},
            vmin=0, vmax=100)
ax.set_title('Customer Cohort Retention Matrix (% of cohort ordering in month N)', fontsize=13)
ax.set_xlabel('Months Since First Order')
ax.set_ylabel('Acquisition Cohort')
plt.tight_layout()
plt.show()
print(f"Avg Month-1 retention: {pivot[1].mean():.1f}%")

## 2. RFM Segment Distribution

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT rfm_segment,
               COUNT(*) AS customers,
               ROUND(AVG(monetary)::numeric,0) AS avg_ltv,
               ROUND(SUM(monetary)::numeric,0) AS total_revenue,
               ROUND(AVG(recency_days)::numeric,0) AS avg_days_inactive
        FROM mv_customer_rfm
        GROUP BY rfm_segment
        ORDER BY avg_ltv DESC
    '''), conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#27AE60','#2ECC71','#F39C12','#E67E22','#E74C3C','#C0392B','#95A5A6']
axes[0].bar(df['rfm_segment'], df['customers'], color=colors[:len(df)])
axes[0].set_title('Customers per RFM Segment')
axes[0].set_xticklabels(df['rfm_segment'], rotation=30, ha='right')
axes[0].set_ylabel('Customers')

axes[1].bar(df['rfm_segment'], df['avg_ltv'], color=colors[:len(df)])
axes[1].set_title('Avg LTV per RFM Segment (₹)')
axes[1].set_xticklabels(df['rfm_segment'], rotation=30, ha='right')
axes[1].set_ylabel('Avg Lifetime Spend (₹)')
plt.tight_layout()
plt.show()
print(df[['rfm_segment','customers','avg_ltv','total_revenue']].to_string(index=False))

## 3. Churn Risk Distribution

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT
            CASE WHEN recency_days <= 14  THEN 'Active (0-14d)'
                 WHEN recency_days <= 30  THEN 'Cooling (15-30d)'
                 WHEN recency_days <= 60  THEN 'At Risk (31-60d)'
                 WHEN recency_days <= 90  THEN 'Churning (61-90d)'
                 ELSE 'Churned (90d+)'
            END AS segment,
            COUNT(*) AS customers,
            ROUND(SUM(monetary)::numeric,0) AS at_risk_revenue
        FROM mv_customer_rfm
        GROUP BY 1
        ORDER BY MIN(recency_days)
    '''), conn)

colors_map = {'Active (0-14d)': '#27AE60', 'Cooling (15-30d)': '#F39C12',
              'At Risk (31-60d)': '#E67E22', 'Churning (61-90d)': '#E74C3C',
              'Churned (90d+)': '#C0392B'}
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df['segment'], df['customers'],
              color=[colors_map.get(s, '#95A5A6') for s in df['segment']])
for bar, rev in zip(bars, df['at_risk_revenue']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f'₹{rev/1000:.0f}K', ha='center', va='bottom', fontsize=9)
ax.set_title('Customer Churn Risk Segments (revenue label = GMV at risk)')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()